# Pong policy evaluation -> TensorBoard
Evaluate the DIAMOND teacher in Pong at **T=1 (raw policy)** and **T=2 (behavior policy)**, log to TensorBoard. Run this notebook from the repo root. All eval logic is in `pong_action_gate/eval/pong_eval.py`; here we only do TensorBoard plumbing.

In [ ]:
# deps: tensorboard (scalars/video) + moviepy (needed by add_video)
!pip -q install tensorboard moviepy >/dev/null 2>&1 || true
import os, sys
# make sure the repo root is importable (edit if you launch jupyter elsewhere)
if not os.path.isdir('pong_action_gate'):
    os.chdir(os.path.expanduser('D:/Users/trhua/Research/Goal-Conditioned-Confounded-MsPacman'))
print('cwd:', os.getcwd())

In [ ]:
import torch, numpy as np
from torch.utils.tensorboard import SummaryWriter
from pong_action_gate.eval.pong_eval import evaluate, ACTION_NAMES

LOGDIR = 'runs/pong_eval'
N_EPISODES = 10        # bump for tighter estimates (slower; ~25-30s/game on CPU)
writer = SummaryWriter(LOGDIR)
for T, tag in [(1.0, 'eval_T1'), (2.0, 'eval_T2')]:
    print(f'=== evaluating temperature {T} ===')
    m, video = evaluate(n_episodes=N_EPISODES, temperature=T)
    step = 0
    for k in ['episode_return', 'score_difference', 'cumulative_reward',
              'episode_length', 'goal_success', 'goal_distance']:
        writer.add_scalar(f'{tag}/{k}', m[k], step)
    # action histogram: per-action frequency (readable bars)
    writer.add_scalars(f'{tag}/action_freq', {n: f for n, f in zip(ACTION_NAMES, m['action_freq'])}, step)
    # rollout video of the 1st game: (T,H,W,3) -> (1,T,3,H,W) uint8
    if video is not None:
        vid = torch.from_numpy(video).permute(0, 3, 1, 2).unsqueeze(0)
        writer.add_video(f'{tag}/rollout_video', vid, step, fps=30)
    print({k: round(m[k], 3) for k in ['episode_return', 'episode_length', 'goal_success', 'goal_distance']},
          'action_freq=', dict(zip(ACTION_NAMES, m['action_freq'])))
writer.close()
print('logged ->', LOGDIR)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs